In [7]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### FIX ME #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter

import pandas as pd

db = AnimalShelter()

df = pd.DataFrame.from_records(db.read({}))

if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)

print("Number of records:", len(df))
print(df.head())
print(df.columns.tolist())

Number of records: 10002
   rec_num age_upon_outcome animal_id animal_type                     breed   
0      2.0           1 year   A725717         Cat    Domestic Shorthair Mix  \
1      3.0          2 years   A716330         Dog   Chihuahua Shorthair Mix   
2      4.0         7 months   A733653         Cat               Siamese Mix   
3      5.0          2 years   A691584         Dog    Labrador Retriever Mix   
4      6.0          5 years   A696004         Dog  Cardigan Welsh Corgi Mix   

          color date_of_birth             datetime            monthyear   
0  Silver Tabby    2015-05-02  2016-05-06 10:49:00  2016-05-06T10:49:00  \
1   Brown/White    2013-11-18  2015-12-28 18:43:00  2015-12-28T18:43:00   
2    Seal Point    2016-01-25  2016-08-27 18:11:00  2016-08-27T18:11:00   
3     Tan/White    2012-11-06  2015-05-30 13:48:00  2015-05-30T13:48:00   
4   Sable/White    2010-01-27  2015-01-28 10:39:00  2015-01-28T10:39:00   

    name outcome_subtype     outcome_type sex_upo

In [8]:
#########################
# Dashboard Layout / View
#########################

app = JupyterDash(__name__)

# Grazioso Salvare logo
image_filename = 'Grazioso Salvare Logo.png'

encoded_image = base64.b64encode(
    open(image_filename, 'rb').read()
).decode()

app.layout = html.Div([

    # Header
    html.Div([

        html.A(
            html.Img(
                src='data:image/png;base64,{}'.format(encoded_image),
                style={
                    'height': '100px',
                    'width': 'auto'
                }
            ),
            href='https://www.snhu.edu',
            target='_blank'
        ),

        html.H1(
            'Grazioso Salvare Animal Rescue Dashboard',
            style={
                'display': 'inline-block',
                'margin-left': '20px'
            }
        ),

        html.H4('Created by: AHMED AHMED')

    ]),

    html.Hr(),

    # Filter
    html.Div([

        html.Label(
            'Select Rescue Type:',
            style={
                'fontWeight': 'bold'
            }
        ),

        dcc.Dropdown(
            id='filter-type',

            options=[
                {
                    'label': 'Water Rescue',
                    'value': 'water'
                },
                {
                    'label': 'Mountain or Wilderness Rescue',
                    'value': 'mountain'
                },
                {
                    'label': 'Disaster or Individual Tracking',
                    'value': 'disaster'
                },
                {
                    'label': 'Reset',
                    'value': 'reset'
                }
            ],

            value='reset',
            clearable=False
        )

    ]),

    html.Hr(),

    # Data table
    dash_table.DataTable(
        id='datatable-id',

        columns=[
            {
                "name": i,
                "id": i,
                "deletable": False,
                "selectable": True
            }
            for i in df.columns
        ],

        data=df.to_dict('records'),

        page_size=10,

        sort_action='native',

        row_selectable='single',

        selected_rows=[],

        style_table={
            'overflowX': 'auto'
        },

        style_cell={
            'textAlign': 'left',
            'padding': '8px',
            'whiteSpace': 'normal',
            'height': 'auto'
        },

        style_header={
            'fontWeight': 'bold'
        }
    ),

    html.Br(),

    html.Hr(),

    # Charts and map
    html.Div(
        className='row',

        style={
            'display': 'flex'
        },

        children=[

            html.Div(
                id='graph-id',
                className='col s12 m6',

                style={
                    'width': '50%'
                }
            ),

            html.Div(
                id='map-id',
                className='col s12 m6',

                style={
                    'width': '50%'
                }
            )

        ]
    )

])

In [9]:
#############################################
# Interaction Between Components / Controller
#############################################

# Update the data table based on the selected rescue type
@app.callback(
    Output('datatable-id', 'data'),
    Input('filter-type', 'value')
)
def update_dashboard(filter_type):

    if filter_type == 'water':
        records = db.water_rescue()

    elif filter_type == 'mountain':
        records = db.mountain_rescue()

    elif filter_type == 'disaster':
        records = db.disaster_rescue()

    else:
        # Reset - return all records
        records = db.read({})

    filtered_df = pd.DataFrame.from_records(records)

    if '_id' in filtered_df.columns:
        filtered_df.drop(columns=['_id'], inplace=True)

    return filtered_df.to_dict('records')


# Update pie chart based on the currently displayed table data
@app.callback(
    Output('graph-id', 'children'),
    Input('datatable-id', 'derived_virtual_data')
)
def update_graphs(viewData):

    if viewData is None:
        dff = df.copy()
    else:
        dff = pd.DataFrame.from_dict(viewData)

    # Make sure there is data to display
    if dff.empty:
        return [
            dcc.Graph(
                figure=px.pie(
                    title='No Animals Found'
                )
            )
        ]

    figure = px.pie(
        dff,
        names='breed',
        title='Animals by Breed'
    )

    return [
        dcc.Graph(
            figure=figure
        )
    ]


# Highlight selected columns in the data table
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    Input('datatable-id', 'selected_columns')
)
def update_styles(selected_columns):

    if selected_columns is None:
        return []

    return [
        {
            'if': {
                'column_id': column
            },
            'background_color': '#D2F3FF'
        }
        for column in selected_columns
    ]


# Update the map based on the selected animal
@app.callback(
    Output('map-id', 'children'),
    [
        Input('datatable-id', 'derived_virtual_data'),
        Input('datatable-id', 'derived_virtual_selected_rows')
    ]
)
def update_map(viewData, index):

    if viewData is None:
        return []

    if index is None or len(index) == 0:
        return []

    dff = pd.DataFrame.from_dict(viewData)

    if dff.empty:
        return []

    row = index[0]

    latitude = dff.iloc[row]['location_lat']
    longitude = dff.iloc[row]['location_long']
    breed = dff.iloc[row]['breed']
    name = dff.iloc[row]['name']

    return [
        dl.Map(
            style={
                'width': '100%',
                'height': '500px'
            },
            center=[latitude, longitude],
            zoom=10,
            children=[
                dl.TileLayer(
                    id="base-layer-id"
                ),

                dl.Marker(
                    position=[
                        latitude,
                        longitude
                    ],

                    children=[
                        dl.Tooltip(
                            breed
                        ),

                        dl.Popup([
                            html.H1(
                                "Animal Name"
                            ),

                            html.P(
                                name
                            ),

                            html.P(
                                breed
                            )
                        ])
                    ]
                )
            ]
        )
    ]


# Run the dashboard
app.run_server()

Dash app running on https://avatarpegasus-airlineaztec-3000.codio.io/proxy/8050/
